# Out-of-Core Processing & Streaming Tutorial

This tutorial illustrates out-of-core spatial data processing on datasets larger than RAM using **Dask** and **Zarr**.

In [ ]:
import spatial_pipeline as sp
import numpy as np

# Create synthetic large dataset
csv_path = "massive_spatial_data.csv"
sp.generate_vectorized_mock_data(csv_path, num_molecules=50000)

## 1. Dask Distributed Spatial Binning

Partition and bin massive transcript coordinates in parallel out-of-core.

In [ ]:
sparse_counts, bin_centers, unique_labels = sp.dask_bin_transcripts(
    csv_path,
    resolution=20.0,
    min_counts=2,
    blocksize="32MB"
)
print(f"Stitched Active Bins: {sparse_counts.shape[0]:,}")
print(f"Unique Labels:         {len(unique_labels)}")

## 2. Chunked Zarr Hierarchy Export & Streaming

In [ ]:
adata = sp.to_anndata(sparse_counts, bin_centers, labels=unique_labels)
zarr_store = "massive_spatial.zarr"
sp.to_zarr(adata, zarr_store, chunks=(2000, len(unique_labels)))

# Stream batches for model training without loading full dataset into RAM
batch_idx = 0
for batch_X, batch_pos, var_names in sp.stream_zarr_anndata(zarr_store, chunk_size=2000):
    batch_idx += 1
    print(f"Batch {batch_idx}: Shape = {batch_X.shape}, Position Shape = {batch_pos.shape}")

## 3. Parallel Dask Array Matrix Metrics

In [ ]:
metrics = sp.dask_compute_spatial_metrics(zarr_store, chunk_size=2000)
print("Mean Label Expression:", metrics["mean"])
print("Label Variance:       ", metrics["var"])
print("Label Sparsity:       ", metrics["sparsity"])